# Anime Colab Studio — móvil (iPhone / Android)

Versión sin widgets: utiliza formularios nativos de Colab, que suelen ser más cómodos en pantallas táctiles. Puedes alternar generación de **imágenes** (SDXL/Illustrious/Pony) y **videos** (Wan 2.2 en ComfyUI o familia Wan/LTX/Hunyuan desde WanGP). Los motores funcionan en la GPU de **Colab**, no en el teléfono.

**Pasos:** GPU activada → celda 1 → celda 2 (seleccionar parámetros) → celda 3 (URL/contraseña) → celda 4 (supervisión opcional). Para dejar de usarlo, en la celda 3 activa `DETENER_SERVIDOR`.

**Importante:** Safari/Chrome en segundo plano pueden desconectar la vista del notebook; la VM de Colab también tiene límites de tiempo y consumo. No garantizamos ejecución indefinida. Guarda resultados en Drive, no compartas la URL ni la contraseña.


In [ ]:
# @title 1. Preparar Colab y Google Drive
%pip -q install -U huggingface_hub requests 'aiohttp>=3.10,<4'
from pathlib import Path
import subprocess, sys, importlib
project = Path('/content/ai_nsfw_source')
if not (project / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Mairon31/ai_nsfw.git', str(project)], check=True)
else:
    subprocess.run(['git', '-C', str(project), 'fetch', '--depth', '1', 'origin', 'master'], check=True)
    subprocess.run(['git', '-C', str(project), 'reset', '--hard', 'origin/master'], check=True)
sys.path.insert(0, str(project))
importlib.invalidate_caches()
for key in list(sys.modules):
    if key == 'studio' or key.startswith('studio.'):
        del sys.modules[key]
from studio import launcher, runtime, video
launcher.mount_drive()
print('Preparado. Abre la celda 2 y elige Imagen o Video.')


In [ ]:
# @title 2. Configuración y arranque (toca el botón de ejecutar)
MODO = 'Imágenes' # @param ['Imágenes', 'Videos']
MODELO_IMAGEN = 'WAI-NSFW-Illustrious v12 (legacy)' # @param ['NoobAI-XL 1.0', 'Pony Diffusion V6 XL', 'WAI-Illustrious SDXL v17', 'WAI-NSFW-Illustrious v12 (legacy)', 'Animagine XL 4.0 Opt', 'Illustrious XL 1.0']
INTERFAZ_IMAGEN = 'AUTOMATIC1111' # @param ['ComfyUI', 'AUTOMATIC1111']
MODELO_VIDEO = 'Wan 2.2 TI2V 5B · ComfyUI' # @param ['Wan 2.2 TI2V 5B · ComfyUI', 'Wan 2.2 T2V / I2V · WanGP', 'Wan 2.1 1.3B · WanGP', 'LTX-2.x · WanGP', 'HunyuanVideo · WanGP']
TUNEL = 'Cloudflare' # @param ['Cloudflare', 'ngrok', 'Local only']
CALIDAD_IMAGEN = 'Balanced' # @param ['Balanced', 'High quality', 'Low VRAM']
LORA_PRESET = 'Ninguno' # @param ['Ninguno', 'Anime Style XL (general SDXL)', 'Sensual Face v3 Type 1 (Illustrious)', 'Sensual Face v3 Type 2 (Illustrious)', 'Sensual Face v2 Type 1 (Pony)', 'Sensual Face v2 Type 2 (Pony)']
LORAS_EXTRA = '' # @param {type:'string'}
ADetailer = False # @param {type:'boolean'}
GUARDAR_EN_DRIVE = True # @param {type:'boolean'}
CACHE_MODELOS = True # @param {type:'boolean'}

from studio import launcher, runtime, video
mode = 'video' if MODO == 'Videos' else 'image'
if MODELO_VIDEO not in video.VIDEO_PRESETS or MODELO_IMAGEN not in launcher.MODELS:
    raise ValueError('Modelo desconocido: recarga la celda 1 para actualizar el catálogo.')
selected_loras = []
if mode == 'image' and LORA_PRESET != 'Ninguno':
    family = launcher.MODELS[MODELO_IMAGEN][2]
    if family not in launcher.LORAS[LORA_PRESET][2]:
        raise ValueError('LoRA incompatible con el modelo seleccionado: ' + LORA_PRESET)
    selected_loras = [LORA_PRESET]
cfg = {**launcher.load_settings(), 'mode': mode, 'video_preset': MODELO_VIDEO,
       'model': MODELO_IMAGEN, 'ui': INTERFAZ_IMAGEN, 'tunnel': TUNEL,
       'profile': CALIDAD_IMAGEN, 'loras': selected_loras,
       'custom_loras': '\n'.join(x.strip() for x in LORAS_EXTRA.split(';') if x.strip()) if mode == 'image' else '',
       'adetailer': ADetailer if mode == 'image' else False,
       'save_to_drive': GUARDAR_EN_DRIVE, 'cache_models': CACHE_MODELOS,
       'allowed_email': ''}
launcher.save_settings(cfg)
secrets = {key: launcher.secret(key) for key in ('HF_TOKEN', 'NGROK_AUTHTOKEN', 'WEBUI_PASSWORD', 'CIVITAI_API_TOKEN')}
pid = runtime.launch_worker(cfg, secrets)
print('Iniciando en la GPU de Colab. PID:', pid)
print('Ejecuta la celda 3 para ver estado, URL y contraseña; repítela hasta que esté listo.')


In [ ]:
# @title 3. Estado, abrir Web UI o detener el servidor
DETENER_SERVIDOR = False # @param {type:'boolean'}
from studio import runtime
from IPython.display import display, HTML
if DETENER_SERVIDOR:
    runtime.stop_session()
    print('Servidor detenido; tus archivos en Drive permanecen.')
else:
    info = runtime.snapshot()
    print('Etapa:', info.get('stage', 'sin iniciar'))
    print('Avance:', info.get('percent', 0), '%')
    print('Mensaje:', info.get('message', ''))
    if info.get('url', '').startswith('https://'):
        from html import escape
        url = escape(info['url'], quote=True)
        display(HTML(f'<p><a href="{url}" target="_blank" rel="noopener">Abrir interfaz de generación</a></p>'))
    if info.get('access_password'):
        print('CONTRASEÑA DE SESIÓN:', info['access_password'])
    elif info.get('password_source') == 'WEBUI_PASSWORD':
        print('Contraseña: usa el secreto WEBUI_PASSWORD que configuraste en Colab.')
    print('Activo:', runtime.process_active())
    print('\nÚltimos registros:')
    print(runtime.recent_log(25))


In [ ]:
# @title 4. Supervisión continua (opcional; deja la celda ejecutándose)
ACTIVAR_MONITOR = True # @param {type:'boolean'}
if ACTIVAR_MONITOR:
    from studio import runtime
    await runtime.monitor_active_session_async(interval=30)
else:
    print('Supervisión desactivada. Puedes consultar el estado en la celda 3.')
